In [8]:
import pandas as pd
import glob

def process_supplier_data(output_file):
    # Define the priority order for trust scores
    trust_score_priority = {'HIGH': 1, 'MEDIUM': 2, 'LOW': 3}
    
    # Define the categories to be processed
    categories = [
        "SBE", "SDB", "MBE", "CAB", "WBE", "WLE", "VBE", "SDVBE", "VOSB", "LGBTBE", "PWD",
        "DBE", "HUB", "BCORP", "DVBE", "HBCU", "HUD"
    ]
    
    # Automatically find the input CSV file that starts with "diversity"
    input_files = glob.glob("diversity*.csv")
    print(f"Found input files: {input_files}")  # Debugging

    if not input_files:
        print("No input file found matching 'diversity*.csv'")
        return

    input_file = input_files[0]  # Use the first matching file
    print(f"Processing file: {input_file}")  # Debugging

    # Read the input CSV file
    df = pd.read_csv(input_file)
    
    print(f"Columns in file: {df.columns}")  # Debugging
    print(df.head())  # Debugging

    # Check if necessary columns exist
    required_columns = {'internal_supplier_id', 'company_name', 'subcategory', 'trust_score'}
    if not required_columns.issubset(set(df.columns)):
        print(f"Missing required columns. Expected at least: {required_columns}")
        return

    # Create an output DataFrame
    output_columns = ["Internal Supplier ID", "Company Name", "Consolidated"] + categories
    output_df = pd.DataFrame(columns=output_columns)
    
    # Get unique supplier IDs
    for supplier_id in df['internal_supplier_id'].unique():
        supplier_data = df[df['internal_supplier_id'] == supplier_id]
        company_name = supplier_data['company_name'].iloc[0]
        consolidated_subcategories = "|".join(supplier_data['subcategory'].dropna().unique())
        
        # Initialize the row dictionary
        row_data = {
            "Internal Supplier ID": supplier_id,
            "Company Name": company_name,
            "Consolidated": consolidated_subcategories
        }
        
        # Assign trust scores based on priority
        for category in categories:
            trust_scores = supplier_data[supplier_data['subcategory'].str.lower() == category.lower()]['trust_score'].dropna()
            if not trust_scores.empty:
                sorted_scores = sorted(trust_scores, key=lambda x: trust_score_priority.get(x, 4))
                row_data[category] = sorted_scores[0]  # Assign highest priority score
            else:
                row_data[category] = ""
        
        # Append row data to DataFrame
        output_df = pd.concat([output_df, pd.DataFrame([row_data])], ignore_index=True)

    # Debugging: Check if the DataFrame has data before saving
    if output_df.empty:
        print("No data processed. Check input file content.")
        return
    else:
        print("Data processed successfully. Writing to file...")

    # Save to output Excel file
    with pd.ExcelWriter(output_file, engine='xlsxwriter') as writer:
        output_df.to_excel(writer, sheet_name='Sheet1', index=False)
        
        # Adjust column widths
        worksheet = writer.sheets['Sheet1']
        worksheet.set_column('A:A', 25)  # Column A width 25
        worksheet.set_column('B:B', 35)  # Column B width 35
        worksheet.set_column('C:C', 30)  # Column C width 30
    
    print(f"Output file saved: {output_file}")

# Example usage
process_supplier_data("output.xlsx")


Found input files: ['diversity_no_spend_export_29f389dd-cf6e-4638-9432-ec1b8d955a28_1738953285922.csv']
Processing file: diversity_no_spend_export_29f389dd-cf6e-4638-9432-ec1b8d955a28_1738953285922.csv
Columns in file: Index(['cert_id', 'subcategory', 'subtype', 'cert_number', 'expiration_date',
       'certifying_body', 'certifying_address', 'certifying_name', 'cert_url',
       'info_message', 'tealbook_id', 'entity_type', 'internal_supplier_id',
       'company_name', 'complete_address', 'trust_score'],
      dtype='object')
                                cert_id subcategory subtype  cert_number  \
0  ec02c175-81f7-4c00-86c6-f3fde6f0dedf         sbe     NaN          NaN   
1  3f0bc11e-48c6-4042-9a94-ad7440c681c5         sbe     NaN          NaN   
2  3f0bc11e-48c6-4042-9a94-ad7440c681c5         sbe     NaN          NaN   
3  ec02c175-81f7-4c00-86c6-f3fde6f0dedf         sbe     NaN          NaN   
4  9e22dbbd-f659-4300-abd7-34832d033884         sbe     NaN          NaN   

  expirat